# KL-IG Evaluation Notebook

## Experiment Design Matrix

**Methods compared:**
| Method | Baseline | Path |
|--------|----------|------|
| Vanilla IG | Zero (black image) | Pixel-space linear |
| Blur IG | Gaussian-blurred image | Pixel-space linear |
| KL-IG | N(0, I) prior | Distribution-space (mu, logvar) |

**Evaluation metrics:**

| Category | Metric | What it measures |
|----------|--------|-----------------|
| **Faithfulness** | Insertion / Deletion curves | Does masking by attribution order change predictions? |
| **Faithfulness** | ROAR (RemOve And Retrain) | Are top-attributed features truly important? |
| **Faithfulness** | Completeness axiom: sum(attr) ~ delta_f | Do attributions sum to the output difference? |
| **Robustness** | Perturbation sigma | Attribution stability under input noise |
| **Robustness** | Variance sigma^2 | Variance of attributions across MC runs |
| **Sensitivity** | Occlusion correlation | Agreement with model-free occlusion importance |
| **Sensitivity** | Sparsity (Gini coefficient) | How concentrated are attributions? |

**KL-IG expected advantages:**
- Noisy inputs -> stochastic smoothing built in
- OOD baselines -> principled N(0,1) reference (no arbitrary zero/blur)
- Completeness -> tighter bounds from KL-path integration
- Stability -> MC averaging reduces run-to-run variance

## 1. Setup & Imports

In [ ]:
import sys
import math
import warnings
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torchvision.models import ResNet50_Weights, resnet50
from scipy import stats
from tqdm.notebook import tqdm

# Ensure repo root is on sys.path
ROOT = Path.cwd()
if not (ROOT / "klig").exists():
    ROOT = ROOT / "infocube-main"
sys.path.append(str(ROOT))

from klig.image.attribution import ImageAttributor
from klig.image.stopping import find_sigma_stop
from klig.compare.captum_baselines import run_ig, _absmax_collapse
from captum.attr import IntegratedGradients

warnings.filterwarnings("ignore", category=UserWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Root:   {ROOT}")

## 2. Configuration & Hyperparameters

In [ ]:
# ── Paths ──
IMAGE_PATH = ROOT / "example.jpg"  # <-- CHANGE to your image path(s)
# For multi-image evaluation, put paths in a list:
# IMAGE_PATHS = list((ROOT / "images").glob("*.jpg"))

# ── KLIG hyperparameters ──
N_STEPS = 50          # Integration steps for KL-IG
N_SAMPLES = 10        # MC samples per step
SIGMA_FINAL = 1/256   # Default final sigma
ADAPTIVE_SIGMA = True # Use binary-search sigma_stop

# ── Baseline hyperparameters ──
IG_STEPS = 50         # Steps for Vanilla IG and Blur IG
BLUR_SIGMA = 16.0     # Gaussian blur kernel sigma for Blur IG baseline
BLUR_KERNEL = 51      # Kernel size (must be odd)

# ── Evaluation settings ──
N_INSERTION_STEPS = 100     # Steps for insertion/deletion curves
N_ROBUSTNESS_RUNS = 10      # MC runs for variance estimation
PERTURBATION_SIGMAS = [0.01, 0.02, 0.05, 0.1, 0.2]  # Noise levels for robustness
OCCLUSION_PATCH_SIZE = 14   # Patch size for occlusion sensitivity
OCCLUSION_STRIDE = 7        # Stride for occlusion patches
CLIP_PCT = 99.0

# ── ImageNet preprocessing ──
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMAGENET_TRANSFORM = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Configuration loaded.")

## 3. Model & Data Loading

In [ ]:
def load_model():
    """Load pretrained ResNet50 (ImageNet V2 weights)."""
    weights = ResNet50_Weights.IMAGENET1K_V2
    model = resnet50(weights=weights).to(DEVICE).eval()
    return model, weights.meta["categories"]

def load_image(path):
    """Load and preprocess a single image -> (1, 3, 224, 224)."""
    img = Image.open(path).convert("RGB")
    return IMAGENET_TRANSFORM(img).unsqueeze(0).to(DEVICE)

def predict_topk(model, x, k=5):
    """Get top-k predictions as [(label, prob), ...], [idx, ...]."""
    with torch.no_grad():
        probs = model(x).softmax(dim=-1)[0]
        top_p, top_i = probs.topk(k)
    labels = ResNet50_Weights.IMAGENET1K_V2.meta["categories"]
    return [(labels[int(i)], float(p)) for i, p in zip(top_i, top_p)], top_i.tolist()

def denormalize(x):
    """Undo ImageNet normalization: (C,H,W) or (1,C,H,W) -> [0,1] range."""
    mean = torch.tensor(IMAGENET_MEAN, device=x.device).view(-1, 1, 1)
    std  = torch.tensor(IMAGENET_STD, device=x.device).view(-1, 1, 1)
    if x.dim() == 4:
        mean, std = mean.unsqueeze(0), std.unsqueeze(0)
    return (x * std + mean).clamp(0, 1)

# Load model
model, imagenet_labels = load_model()
print(f"Model loaded: ResNet50 ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)")

# Load image
x = load_image(IMAGE_PATH)
preds, top_idx = predict_topk(model, x)
target_idx = top_idx[0]

print(f"\nImage: {IMAGE_PATH.name}")
print(f"Target class: {preds[0][0]} (idx={target_idx}, prob={preds[0][1]:.4f})")
print(f"\nTop-5 predictions:")
for label, prob in preds:
    print(f"  {label}: {prob:.4f}")

# Show the input image
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(denormalize(x[0]).cpu().permute(1, 2, 0).numpy())
ax.set_title(f"{preds[0][0]} ({preds[0][1]:.1%})")
ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Attribution Methods

Three methods are compared:
1. **Vanilla IG** -- zero (black) baseline, linear pixel-space interpolation
2. **Blur IG** -- Gaussian-blurred baseline, linear pixel-space interpolation  
3. **KL-IG** -- stochastic path through (mu, logvar) distribution space

In [ ]:
# ─────────────────────────────────────────────
# 4a. Blur IG implementation (not in repo yet)
# ─────────────────────────────────────────────

def make_gaussian_blur_baseline(x, kernel_size=BLUR_KERNEL, sigma=BLUR_SIGMA):
    """Create a Gaussian-blurred version of x as the Blur IG baseline.
    
    Args:
        x: (1, C, H, W) input tensor
        kernel_size: odd int, blur kernel size
        sigma: float, Gaussian sigma
    Returns:
        (1, C, H, W) blurred tensor
    """
    # Build 1D Gaussian kernel
    coords = torch.arange(kernel_size, dtype=torch.float32, device=x.device) - kernel_size // 2
    kernel_1d = torch.exp(-0.5 * (coords / sigma) ** 2)
    kernel_1d = kernel_1d / kernel_1d.sum()
    
    # Separable 2D convolution per channel
    kernel_h = kernel_1d.view(1, 1, -1, 1).expand(3, -1, -1, -1)
    kernel_w = kernel_1d.view(1, 1, 1, -1).expand(3, -1, -1, -1)
    
    pad = kernel_size // 2
    blurred = F.conv2d(x, kernel_h, padding=(pad, 0), groups=3)
    blurred = F.conv2d(blurred, kernel_w, padding=(0, pad), groups=3)
    return blurred


def run_blur_ig(model, x, target, n_steps=IG_STEPS, blur_sigma=BLUR_SIGMA):
    """Integrated Gradients with Gaussian-blur baseline.
    
    Returns:
        (H, W) attribution map (abs-max collapsed).
    """
    baseline = make_gaussian_blur_baseline(x, sigma=blur_sigma)
    return run_ig(model, x, target, n_steps=n_steps, baseline=baseline)


# ─────────────────────────────────────────────
# 4b. Convenience wrappers for all 3 methods
# ─────────────────────────────────────────────

def compute_vanilla_ig(model, x, target, n_steps=IG_STEPS):
    """Vanilla IG (zero baseline) -> (H, W) map."""
    return run_ig(model, x, target, n_steps=n_steps)


def compute_blur_ig(model, x, target, n_steps=IG_STEPS, blur_sigma=BLUR_SIGMA):
    """Blur IG (Gaussian blur baseline) -> (H, W) map."""
    return run_blur_ig(model, x, target, n_steps=n_steps, blur_sigma=blur_sigma)


def compute_klig(model, x, target, n_steps=N_STEPS, n_samples=N_SAMPLES,
                 sigma_final=SIGMA_FINAL, adaptive=ADAPTIVE_SIGMA):
    """KL-IG (stochastic distribution path) -> ImageAttributionResult."""
    if adaptive:
        sigma_final = find_sigma_stop(model, x, target=target, tau=0.95)
        sigma_final = max(sigma_final, 1.0 / 256.0)
    
    attributor = ImageAttributor(
        model=model, n_steps=n_steps, n_samples=n_samples,
        sigma_final=sigma_final, device=DEVICE,
    )
    return attributor.attribute(x, target=target, show_progress=True)


def compute_all_attributions(model, x, target):
    """Run all 3 methods. Returns dict of (H, W) maps + KL-IG result object."""
    print("Computing Vanilla IG...")
    vanilla_map = compute_vanilla_ig(model, x, target)
    
    print("Computing Blur IG...")
    blur_map = compute_blur_ig(model, x, target)
    
    print("Computing KL-IG...")
    klig_result = compute_klig(model, x, target)
    klig_map = klig_result.attr_map("absmax")
    
    maps = {
        "Vanilla IG": vanilla_map,
        "Blur IG": blur_map,
        "KL-IG": klig_map,
    }
    return maps, klig_result

print("Attribution functions defined.")

In [ ]:
# ─────────────────────────────────────────────
# 4c. Compute attributions for our test image
# ─────────────────────────────────────────────

attr_maps, klig_result = compute_all_attributions(model, x, target_idx)

# Quick visual comparison
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Original image
axes[0].imshow(denormalize(x[0]).cpu().permute(1, 2, 0).numpy())
axes[0].set_title("Original", fontsize=11)
axes[0].axis("off")

# Attribution maps
from klig.image.viz import _attr_to_rgb

for i, (name, amap) in enumerate(attr_maps.items()):
    axes[i + 1].imshow(_attr_to_rgb(amap, clip_percentile=CLIP_PCT))
    axes[i + 1].set_title(name, fontsize=11)
    axes[i + 1].axis("off")

fig.suptitle(f"Attribution Maps for: {preds[0][0]}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Print completeness for KL-IG
print(f"\nKL-IG completeness check: sum(attr) = {klig_result._r.completeness_check():.4f}")

---
## 5. Faithfulness Metrics

### 5a. Insertion & Deletion Curves

**Insertion**: Starting from a blank image, progressively reveal pixels in order of attribution importance. A faithful method should cause the prediction to rise quickly.

**Deletion**: Starting from the original image, progressively remove pixels in order of attribution importance. A faithful method should cause the prediction to drop quickly.

The **AUC** (area under curve) is the summary metric: higher insertion AUC = better, lower deletion AUC = better.

In [ ]:
def insertion_deletion_curves(model, x, attr_map, target, n_steps=N_INSERTION_STEPS,
                              substrate_fn="blur"):
    """Compute insertion and deletion curves for a (H, W) attribution map.
    
    Args:
        model: classifier
        x: (1, C, H, W) input image
        attr_map: (H, W) attribution map
        target: class index
        n_steps: number of steps in the curve
        substrate_fn: "blur" or "zero" -- what to replace pixels with
        
    Returns:
        dict with 'insertion' and 'deletion' arrays of shape (n_steps+1,)
        and their AUCs
    """
    model.eval()
    H, W = attr_map.shape
    n_pixels = H * W
    
    # Flatten and sort pixels by attribution magnitude (descending)
    attr_flat = attr_map.detach().cpu().abs().view(-1)
    sorted_idx = attr_flat.argsort(descending=True)
    
    # Create substrate (replacement image)
    if substrate_fn == "blur":
        substrate = make_gaussian_blur_baseline(x)
    else:
        substrate = torch.zeros_like(x)
    
    # Pixels to reveal/remove at each step
    pixels_per_step = max(1, n_pixels // n_steps)
    
    insertion_scores = []
    deletion_scores = []
    
    # Start states
    insert_img = substrate.clone()  # start from substrate
    delete_img = x.clone()          # start from original
    
    with torch.no_grad():
        # Score at step 0
        insertion_scores.append(model(insert_img).softmax(dim=-1)[0, target].item())
        deletion_scores.append(model(delete_img).softmax(dim=-1)[0, target].item())
        
        for step in range(1, n_steps + 1):
            start = (step - 1) * pixels_per_step
            end = min(step * pixels_per_step, n_pixels)
            
            if start >= n_pixels:
                insertion_scores.append(insertion_scores[-1])
                deletion_scores.append(deletion_scores[-1])
                continue
            
            pixel_indices = sorted_idx[start:end]
            
            # Convert flat indices to (h, w) coordinates
            h_idx = pixel_indices // W
            w_idx = pixel_indices % W
            
            # Insertion: copy pixels from original to substrate
            insert_img[0, :, h_idx, w_idx] = x[0, :, h_idx, w_idx]
            insertion_scores.append(model(insert_img).softmax(dim=-1)[0, target].item())
            
            # Deletion: replace pixels with substrate
            delete_img[0, :, h_idx, w_idx] = substrate[0, :, h_idx, w_idx]
            deletion_scores.append(model(delete_img).softmax(dim=-1)[0, target].item())
    
    ins = np.array(insertion_scores)
    dele = np.array(deletion_scores)
    
    return {
        "insertion": ins,
        "deletion": dele,
        "insertion_auc": np.trapezoid(ins, dx=1.0 / n_steps),
        "deletion_auc": np.trapezoid(dele, dx=1.0 / n_steps),
    }

print("Insertion/deletion function defined.")

In [ ]:
# Run insertion/deletion for all 3 methods
print("Running insertion/deletion curves...")
id_results = {}
for name, amap in attr_maps.items():
    print(f"  {name}...")
    id_results[name] = insertion_deletion_curves(model, x, amap, target_idx)

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {"Vanilla IG": "#7B68EE", "Blur IG": "#E07B39", "KL-IG": "#2E8B57"}
frac = np.linspace(0, 1, N_INSERTION_STEPS + 1)

for name, res in id_results.items():
    axes[0].plot(frac, res["insertion"], label=f'{name} (AUC={res["insertion_auc"]:.3f})',
                 color=colors[name], linewidth=2)
    axes[1].plot(frac, res["deletion"], label=f'{name} (AUC={res["deletion_auc"]:.3f})',
                 color=colors[name], linewidth=2)

axes[0].set_title("Insertion Curve (higher = better)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Fraction of pixels inserted")
axes[0].set_ylabel("P(target class)")
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

axes[1].set_title("Deletion Curve (lower = better)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Fraction of pixels deleted")
axes[1].set_ylabel("P(target class)")
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Summary table
print("\n┌─────────────┬────────────────┬────────────────┐")
print("│ Method      │ Insertion AUC  │ Deletion AUC   │")
print("├─────────────┼────────────────┼────────────────┤")
for name, res in id_results.items():
    print(f"│ {name:<11} │ {res['insertion_auc']:>14.4f} │ {res['deletion_auc']:>14.4f} │")
print("└─────────────┴────────────────┴────────────────┘")

### 5b. ROAR -- RemOve And Retrain (Approximation)

Full ROAR retrains the model after removing top-k% attributed features. Here we use **ROAR-lite** (no retraining): remove top-k% pixels and measure accuracy drop. This is a lightweight proxy that still measures whether top-attributed pixels are genuinely important.

In [ ]:
def roar_lite(model, x, attr_map, target, percentages=None):
    """ROAR-lite: remove top-k% attributed pixels and measure logit drop.
    
    Args:
        model: classifier
        x: (1, C, H, W) input
        attr_map: (H, W) attribution map
        target: class index
        percentages: list of removal fractions (0 to 1)
    
    Returns:
        dict with 'percentages' and 'logit_drops' and 'prob_remaining'
    """
    if percentages is None:
        percentages = [0.0, 0.05, 0.10, 0.20, 0.30, 0.50, 0.70, 0.90, 1.0]
    
    model.eval()
    H, W = attr_map.shape
    n_pixels = H * W
    
    attr_flat = attr_map.detach().cpu().abs().view(-1)
    sorted_idx = attr_flat.argsort(descending=True)
    
    # Baseline: blurred substrate
    substrate = make_gaussian_blur_baseline(x)
    
    with torch.no_grad():
        clean_logit = model(x)[0, target].item()
        clean_prob = model(x).softmax(dim=-1)[0, target].item()
    
    logit_drops = []
    prob_remaining = []
    
    with torch.no_grad():
        for pct in percentages:
            n_remove = int(pct * n_pixels)
            masked = x.clone()
            
            if n_remove > 0:
                remove_idx = sorted_idx[:n_remove]
                h_idx = remove_idx // W
                w_idx = remove_idx % W
                masked[0, :, h_idx, w_idx] = substrate[0, :, h_idx, w_idx]
            
            logit = model(masked)[0, target].item()
            prob = model(masked).softmax(dim=-1)[0, target].item()
            logit_drops.append(clean_logit - logit)
            prob_remaining.append(prob)
    
    return {
        "percentages": np.array(percentages),
        "logit_drops": np.array(logit_drops),
        "prob_remaining": np.array(prob_remaining),
    }


# Run ROAR-lite
print("Running ROAR-lite...")
roar_results = {}
for name, amap in attr_maps.items():
    roar_results[name] = roar_lite(model, x, amap, target_idx)

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, res in roar_results.items():
    axes[0].plot(res["percentages"] * 100, res["logit_drops"],
                 label=name, color=colors[name], linewidth=2, marker="o", markersize=4)
    axes[1].plot(res["percentages"] * 100, res["prob_remaining"],
                 label=name, color=colors[name], linewidth=2, marker="o", markersize=4)

axes[0].set_title("ROAR-lite: Logit Drop (higher = more faithful)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("% of top pixels removed")
axes[0].set_ylabel("Logit drop from clean")
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

axes[1].set_title("ROAR-lite: Remaining Probability", fontsize=12, fontweight="bold")
axes[1].set_xlabel("% of top pixels removed")
axes[1].set_ylabel("P(target class)")
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 5c. Completeness Axiom: sum(attr) ~ delta_f

A core axiom of Integrated Gradients: the sum of all attributions should equal the difference between the model output at the input and the model output at the baseline.

- **Vanilla IG**: sum(attr) = f(x) - f(zero_baseline)
- **Blur IG**: sum(attr) = f(x) - f(blur_baseline)
- **KL-IG**: sum(attr) ~ E[f(x_final)] - E[f(x_noise)]

KL-IG's stochastic completeness should yield **tighter bounds** since the reference is principled.

In [ ]:
def completeness_check(model, x, attr_maps_chw, target):
    """Check completeness axiom for all methods.
    
    Args:
        model: classifier
        x: (1, C, H, W) input
        attr_maps_chw: dict of {name: (C,H,W) or (H,W) attribution tensors}
        target: class index
    
    Returns:
        dict of {name: {sum_attr, delta_f, completeness_gap, relative_error}}
    """
    model.eval()
    
    with torch.no_grad():
        f_input = model(x)[0, target].item()
    
    results = {}
    
    # Vanilla IG: baseline = zeros
    zero_baseline = torch.zeros_like(x)
    with torch.no_grad():
        f_zero = model(zero_baseline)[0, target].item()
    delta_f_vanilla = f_input - f_zero
    
    # We need (C,H,W) attributions for proper sum -- recompute with Captum raw output
    ig = IntegratedGradients(model)
    vanilla_attr_chw = ig.attribute(x, baselines=zero_baseline, target=target,
                                     n_steps=IG_STEPS, method="gausslegendre")
    sum_vanilla = vanilla_attr_chw.detach().sum().item()
    
    results["Vanilla IG"] = {
        "sum_attr": sum_vanilla,
        "delta_f": delta_f_vanilla,
        "gap": abs(sum_vanilla - delta_f_vanilla),
        "relative_error": abs(sum_vanilla - delta_f_vanilla) / (abs(delta_f_vanilla) + 1e-10),
    }
    
    # Blur IG: baseline = blurred image
    blur_baseline = make_gaussian_blur_baseline(x)
    with torch.no_grad():
        f_blur = model(blur_baseline)[0, target].item()
    delta_f_blur = f_input - f_blur
    
    blur_attr_chw = ig.attribute(x, baselines=blur_baseline, target=target,
                                  n_steps=IG_STEPS, method="gausslegendre")
    sum_blur = blur_attr_chw.detach().sum().item()
    
    results["Blur IG"] = {
        "sum_attr": sum_blur,
        "delta_f": delta_f_blur,
        "gap": abs(sum_blur - delta_f_blur),
        "relative_error": abs(sum_blur - delta_f_blur) / (abs(delta_f_blur) + 1e-10),
    }
    
    # KL-IG: completeness from the result object
    sum_klig = klig_result._r.completeness_check()
    # delta_f for KL-IG: E[f(x_final)] - E[f(x_noise)]
    # Approximate E[f(noise)] with MC samples from N(0,I)
    n_mc = 100
    with torch.no_grad():
        noise = torch.randn(n_mc, *x.squeeze(0).shape, device=DEVICE)
        f_noise = model(noise)[:, target].mean().item()
    delta_f_klig = f_input - f_noise
    
    results["KL-IG"] = {
        "sum_attr": sum_klig,
        "delta_f": delta_f_klig,
        "gap": abs(sum_klig - delta_f_klig),
        "relative_error": abs(sum_klig - delta_f_klig) / (abs(delta_f_klig) + 1e-10),
    }
    
    return results


# Run completeness check
print("Checking completeness axiom...")
comp_results = completeness_check(model, x, attr_maps, target_idx)

# Display results
print("\n┌─────────────┬───────────────┬───────────────┬───────────────┬───────────────┐")
print("│ Method      │   sum(attr)   │    delta_f    │     gap       │  rel. error   │")
print("├─────────────┼───────────────┼───────────────┼───────────────┼───────────────┤")
for name, res in comp_results.items():
    print(f"│ {name:<11} │ {res['sum_attr']:>13.4f} │ {res['delta_f']:>13.4f} │ "
          f"{res['gap']:>13.4f} │ {res['relative_error']:>12.4%} │")
print("└─────────────┴───────────────┴───────────────┴───────────────┴───────────────┘")

# Bar chart of relative completeness errors
fig, ax = plt.subplots(figsize=(8, 4))
names = list(comp_results.keys())
errors = [comp_results[n]["relative_error"] * 100 for n in names]
bars = ax.bar(names, errors, color=[colors[n] for n in names], edgecolor="black", linewidth=0.5)
ax.set_ylabel("Relative Completeness Error (%)")
ax.set_title("Completeness Axiom: sum(attr) vs delta_f\n(lower = better)", fontsize=12, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
for bar, err in zip(bars, errors):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{err:.2f}%", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

---
## 6. Robustness Metrics

### 6a. Perturbation Sensitivity (sigma)

How stable are attributions when the input is slightly perturbed? For each noise level sigma, we add Gaussian noise to the input, recompute attributions, and measure the Spearman rank correlation with the clean attribution.

KL-IG should be more robust due to its **built-in stochastic smoothing** from MC sampling.

In [ ]:
def perturbation_robustness(model, x, target, sigmas=PERTURBATION_SIGMAS, n_runs=3):
    """Measure attribution stability under input perturbation.
    
    For each noise level sigma, adds N(0, sigma^2) noise to x,
    recomputes attributions, and measures rank correlation with the clean attribution.
    
    Returns:
        dict of {method_name: {sigma: mean_spearman_corr}}
    """
    model.eval()
    
    # Clean attributions
    clean_maps = {
        "Vanilla IG": compute_vanilla_ig(model, x, target).detach().cpu().numpy().ravel(),
        "Blur IG": compute_blur_ig(model, x, target).detach().cpu().numpy().ravel(),
    }
    klig_res = compute_klig(model, x, target)
    clean_maps["KL-IG"] = klig_res.attr_map("absmax").detach().cpu().numpy().ravel()
    
    results = {name: {} for name in clean_maps}
    
    for sigma in sigmas:
        print(f"  sigma={sigma:.3f}...")
        for name in clean_maps:
            corrs = []
            for _ in range(n_runs):
                # Perturbed input
                x_noisy = x + sigma * torch.randn_like(x)
                
                if name == "Vanilla IG":
                    noisy_map = compute_vanilla_ig(model, x_noisy, target)
                elif name == "Blur IG":
                    noisy_map = compute_blur_ig(model, x_noisy, target)
                else:
                    noisy_result = compute_klig(model, x_noisy, target)
                    noisy_map = noisy_result.attr_map("absmax")
                
                noisy_flat = noisy_map.detach().cpu().numpy().ravel()
                corr, _ = stats.spearmanr(clean_maps[name], noisy_flat)
                corrs.append(corr)
            
            results[name][sigma] = np.mean(corrs)
    
    return results


print("Running perturbation robustness...")
perturb_results = perturbation_robustness(model, x, target_idx)

# ── Plot ──
fig, ax = plt.subplots(figsize=(10, 5))
for name in perturb_results:
    sigs = sorted(perturb_results[name].keys())
    corrs = [perturb_results[name][s] for s in sigs]
    ax.plot(sigs, corrs, label=name, color=colors[name], linewidth=2, marker="o", markersize=6)

ax.set_xlabel("Perturbation sigma", fontsize=11)
ax.set_ylabel("Spearman rank correlation with clean", fontsize=11)
ax.set_title("Perturbation Robustness (higher = more stable)", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

### 6b. Variance (sigma^2) -- MC Run-to-Run Stability

Rerun each method multiple times on the **same input** and measure the pixel-wise variance of attributions. KL-IG uses MC sampling internally, so we expect its **MC averaging** to produce stable outputs despite stochasticity.

For Vanilla/Blur IG (deterministic), we add a tiny perturbation epsilon ~ N(0, 0.001) to simulate numerical instability.

In [ ]:
def variance_stability(model, x, target, n_runs=N_ROBUSTNESS_RUNS, epsilon=1e-3):
    """Measure run-to-run variance of attributions.
    
    For stochastic methods (KL-IG): rerun on the same input.
    For deterministic methods (IG): add tiny epsilon perturbation to simulate instability.
    
    Returns:
        dict of {method: {mean_variance, max_variance, coefficient_of_variation, maps}}
    """
    all_maps = {"Vanilla IG": [], "Blur IG": [], "KL-IG": []}
    
    for run in range(n_runs):
        print(f"  Run {run + 1}/{n_runs}...")
        
        # Small perturbation for deterministic methods to measure sensitivity
        x_eps = x + epsilon * torch.randn_like(x)
        
        all_maps["Vanilla IG"].append(
            compute_vanilla_ig(model, x_eps, target).detach().cpu().numpy()
        )
        all_maps["Blur IG"].append(
            compute_blur_ig(model, x_eps, target).detach().cpu().numpy()
        )
        # KL-IG is stochastic by nature -- run on clean input
        klig_res = compute_klig(model, x, target)
        all_maps["KL-IG"].append(
            klig_res.attr_map("absmax").detach().cpu().numpy()
        )
    
    results = {}
    for name, maps_list in all_maps.items():
        stacked = np.stack(maps_list, axis=0)  # (n_runs, H, W)
        pixelwise_var = stacked.var(axis=0)      # (H, W)
        pixelwise_mean = stacked.mean(axis=0)    # (H, W)
        
        # Coefficient of variation: std / |mean| (averaged over pixels)
        with np.errstate(divide="ignore", invalid="ignore"):
            cv = np.where(np.abs(pixelwise_mean) > 1e-10,
                         np.sqrt(pixelwise_var) / np.abs(pixelwise_mean), 0)
        
        results[name] = {
            "mean_variance": pixelwise_var.mean(),
            "max_variance": pixelwise_var.max(),
            "mean_cv": cv.mean(),
            "variance_map": pixelwise_var,
            "maps": stacked,
        }
    
    return results


print("Running variance stability analysis...")
var_results = variance_stability(model, x, target_idx)

# ── Plot: variance heatmaps ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (name, res) in enumerate(var_results.items()):
    im = axes[i].imshow(res["variance_map"], cmap="hot", interpolation="nearest")
    axes[i].set_title(f"{name}\nmean_var={res['mean_variance']:.2e}", fontsize=10)
    axes[i].axis("off")
    plt.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)

fig.suptitle("Pixel-wise Attribution Variance (lower = more stable)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Summary table
print("\n┌─────────────┬───────────────┬───────────────┬───────────────┐")
print("│ Method      │  Mean Var     │   Max Var     │   Mean CV     │")
print("├─────────────┼───────────────┼───────────────┼───────────────┤")
for name, res in var_results.items():
    print(f"│ {name:<11} │ {res['mean_variance']:>13.2e} │ {res['max_variance']:>13.2e} │ {res['mean_cv']:>13.4f} │")
print("└─────────────┴───────────────┴───────────────┴───────────────┘")

---
## 7. Sensitivity Metrics

### 7a. Occlusion Correlation

Occlusion sensitivity is a model-free "ground truth" for feature importance: slide a grey patch over the image and measure how much the output drops at each location. A good attribution method should correlate with this occlusion importance map.

In [ ]:
def occlusion_sensitivity(model, x, target, patch_size=OCCLUSION_PATCH_SIZE,
                          stride=OCCLUSION_STRIDE):
    """Compute occlusion sensitivity map.
    
    Slides a grey patch over the image and measures logit drop at each position.
    
    Returns:
        (H_out, W_out) numpy array of logit drops (higher = more important)
    """
    model.eval()
    _, C, H, W = x.shape
    
    with torch.no_grad():
        f_clean = model(x)[0, target].item()
    
    h_positions = list(range(0, H - patch_size + 1, stride))
    w_positions = list(range(0, W - patch_size + 1, stride))
    
    occ_map = np.zeros((len(h_positions), len(w_positions)))
    
    with torch.no_grad():
        for i, h in enumerate(h_positions):
            for j, w in enumerate(w_positions):
                masked = x.clone()
                masked[0, :, h:h+patch_size, w:w+patch_size] = 0  # grey patch (normalized zero = grey)
                f_masked = model(masked)[0, target].item()
                occ_map[i, j] = f_clean - f_masked  # positive = important
    
    return occ_map


def attribution_to_patch_grid(attr_map, patch_size, stride, H, W):
    """Downsample (H, W) attribution to patch grid by averaging patches."""
    attr_np = attr_map.detach().cpu().abs().numpy()
    h_positions = list(range(0, H - patch_size + 1, stride))
    w_positions = list(range(0, W - patch_size + 1, stride))
    
    grid = np.zeros((len(h_positions), len(w_positions)))
    for i, h in enumerate(h_positions):
        for j, w in enumerate(w_positions):
            grid[i, j] = attr_np[h:h+patch_size, w:w+patch_size].mean()
    return grid


# Compute occlusion sensitivity
print("Computing occlusion sensitivity map...")
occ_map = occlusion_sensitivity(model, x, target_idx)

# Compare each method's attribution with occlusion
_, _, H, W = x.shape
occ_corrs = {}
for name, amap in attr_maps.items():
    attr_grid = attribution_to_patch_grid(amap, OCCLUSION_PATCH_SIZE, OCCLUSION_STRIDE, H, W)
    corr, pval = stats.spearmanr(occ_map.ravel(), attr_grid.ravel())
    occ_corrs[name] = {"spearman": corr, "pval": pval}

# ── Plot ──
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Occlusion map
im = axes[0].imshow(occ_map, cmap="hot", interpolation="nearest")
axes[0].set_title("Occlusion Sensitivity\n(ground truth)", fontsize=10)
axes[0].axis("off")
plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

for i, (name, amap) in enumerate(attr_maps.items()):
    attr_grid = attribution_to_patch_grid(amap, OCCLUSION_PATCH_SIZE, OCCLUSION_STRIDE, H, W)
    im = axes[i + 1].imshow(attr_grid, cmap="hot", interpolation="nearest")
    axes[i + 1].set_title(f"{name}\nrho={occ_corrs[name]['spearman']:.3f}", fontsize=10)
    axes[i + 1].axis("off")
    plt.colorbar(im, ax=axes[i + 1], fraction=0.046, pad=0.04)

fig.suptitle("Occlusion Correlation: Attribution vs Occlusion Sensitivity", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
names = list(occ_corrs.keys())
rhos = [occ_corrs[n]["spearman"] for n in names]
bars = ax.bar(names, rhos, color=[colors[n] for n in names], edgecolor="black", linewidth=0.5)
ax.set_ylabel("Spearman Correlation with Occlusion")
ax.set_title("Occlusion Correlation (higher = better)", fontsize=12, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
for bar, rho in zip(bars, rhos):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{rho:.3f}", ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.show()

### 7b. Sparsity (Gini Coefficient)

The **Gini coefficient** measures how concentrated attributions are. A Gini of 1.0 means all attribution is on a single pixel; 0.0 means uniform spread. Sparser attributions are generally more interpretable and localised.

In [ ]:
def gini_coefficient(values):
    """Compute the Gini coefficient of a 1D array of non-negative values."""
    v = np.sort(np.abs(values.ravel()))
    n = len(v)
    if n == 0 or v.sum() == 0:
        return 0.0
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * v) / (n * np.sum(v))) - (n + 1) / n


def top_k_mass(attr_map, k_percent=10):
    """What fraction of total |attribution| is in the top k% of pixels?"""
    flat = np.abs(attr_map.detach().cpu().numpy().ravel())
    n = len(flat)
    k = max(1, int(k_percent / 100 * n))
    sorted_vals = np.sort(flat)[::-1]
    total = flat.sum()
    if total == 0:
        return 0.0
    return sorted_vals[:k].sum() / total


# Compute sparsity metrics
sparsity_results = {}
for name, amap in attr_maps.items():
    flat = amap.detach().cpu().numpy().ravel()
    sparsity_results[name] = {
        "gini": gini_coefficient(flat),
        "top_5_mass": top_k_mass(amap, k_percent=5),
        "top_10_mass": top_k_mass(amap, k_percent=10),
        "top_20_mass": top_k_mass(amap, k_percent=20),
    }

# ── Table ──
print("┌─────────────┬──────────┬────────────┬─────────────┬─────────────┐")
print("│ Method      │   Gini   │ Top-5% mass│ Top-10% mass│ Top-20% mass│")
print("├─────────────┼──────────┼────────────┼─────────────┼─────────────┤")
for name, res in sparsity_results.items():
    print(f"│ {name:<11} │ {res['gini']:>8.4f} │ {res['top_5_mass']:>10.4f} │ "
          f"{res['top_10_mass']:>11.4f} │ {res['top_20_mass']:>11.4f} │")
print("└─────────────┴──────────┴────────────┴─────────────┴─────────────┘")

# ── Plot: Gini + top-k mass ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gini coefficients
names = list(sparsity_results.keys())
ginis = [sparsity_results[n]["gini"] for n in names]
bars = axes[0].bar(names, ginis, color=[colors[n] for n in names], edgecolor="black", linewidth=0.5)
axes[0].set_ylabel("Gini Coefficient")
axes[0].set_title("Attribution Sparsity (Gini)\nhigher = more concentrated", fontsize=12, fontweight="bold")
axes[0].grid(axis="y", alpha=0.3)
for bar, g in zip(bars, ginis):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f"{g:.4f}", ha="center", va="bottom", fontsize=10)

# Top-k mass comparison
x_pos = np.arange(3)
width = 0.25
for i, name in enumerate(names):
    masses = [sparsity_results[name][f"top_{k}_mass"] for k in [5, 10, 20]]
    axes[1].bar(x_pos + i * width, masses, width, label=name,
                color=colors[name], edgecolor="black", linewidth=0.5)

axes[1].set_xticks(x_pos + width)
axes[1].set_xticklabels(["Top 5%", "Top 10%", "Top 20%"])
axes[1].set_ylabel("Fraction of total |attribution|")
axes[1].set_title("Attribution Mass Concentration", fontsize=12, fontweight="bold")
axes[1].legend(fontsize=9)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# ── Lorenz curves ──
fig, ax = plt.subplots(figsize=(7, 7))
for name, amap in attr_maps.items():
    flat = np.sort(np.abs(amap.detach().cpu().numpy().ravel()))
    cumulative = np.cumsum(flat) / flat.sum()
    frac = np.linspace(0, 1, len(cumulative))
    ax.plot(frac, cumulative, label=f"{name} (Gini={sparsity_results[name]['gini']:.3f})",
            color=colors[name], linewidth=2)

ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Perfect equality")
ax.set_xlabel("Fraction of pixels (sorted by |attr|)")
ax.set_ylabel("Cumulative fraction of total |attr|")
ax.set_title("Lorenz Curves of Attribution Concentration", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

---
## 8. KL-IG Advantage Analysis: OOD Baselines

Vanilla IG uses a **zero (black) baseline**, which is an out-of-distribution input -- the model may have undefined behaviour on it. Blur IG uses a blurred image, which is slightly better. KL-IG uses N(0,1) as a principled information-theoretic reference.

Here we visualise **what each baseline looks like to the model** and how the baseline choice affects the attribution path.

In [ ]:
def analyse_baselines(model, x, target):
    """Analyse the baselines used by each method -- how OOD are they?"""
    model.eval()
    
    with torch.no_grad():
        # Clean prediction
        logits_clean = model(x)[0]
        probs_clean = logits_clean.softmax(dim=-1)
        
        # Zero baseline
        zero = torch.zeros_like(x)
        logits_zero = model(zero)[0]
        probs_zero = logits_zero.softmax(dim=-1)
        entropy_zero = -(probs_zero * probs_zero.log().clamp(min=-100)).sum().item()
        
        # Blur baseline
        blur = make_gaussian_blur_baseline(x)
        logits_blur = model(blur)[0]
        probs_blur = logits_blur.softmax(dim=-1)
        entropy_blur = -(probs_blur * probs_blur.log().clamp(min=-100)).sum().item()
        
        # KL-IG reference: samples from N(0, I)
        n_mc = 50
        noise = torch.randn(n_mc, *x.squeeze(0).shape, device=DEVICE)
        logits_noise = model(noise)
        probs_noise = logits_noise.softmax(dim=-1).mean(dim=0)
        entropy_noise = -(probs_noise * probs_noise.log().clamp(min=-100)).sum().item()
        
        max_entropy = np.log(1000)  # uniform over 1000 ImageNet classes
    
    return {
        "Zero baseline": {
            "target_prob": probs_zero[target].item(),
            "entropy": entropy_zero,
            "entropy_ratio": entropy_zero / max_entropy,
            "top_class": imagenet_labels[probs_zero.argmax().item()],
            "top_prob": probs_zero.max().item(),
        },
        "Blur baseline": {
            "target_prob": probs_blur[target].item(),
            "entropy": entropy_blur,
            "entropy_ratio": entropy_blur / max_entropy,
            "top_class": imagenet_labels[probs_blur.argmax().item()],
            "top_prob": probs_blur.max().item(),
        },
        "N(0,1) reference": {
            "target_prob": probs_noise[target].item(),
            "entropy": entropy_noise,
            "entropy_ratio": entropy_noise / max_entropy,
            "top_class": imagenet_labels[probs_noise.argmax().item()],
            "top_prob": probs_noise.max().item(),
        },
    }


baseline_analysis = analyse_baselines(model, x, target_idx)

# Visualise baselines
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Original
axes[0].imshow(denormalize(x[0]).cpu().permute(1, 2, 0).numpy())
axes[0].set_title(f"Original\nP(target)={preds[0][1]:.3f}", fontsize=10)
axes[0].axis("off")

# Zero baseline
zero_img = torch.zeros_like(x)
axes[1].imshow(denormalize(zero_img[0]).cpu().permute(1, 2, 0).numpy())
ba = baseline_analysis["Zero baseline"]
axes[1].set_title(f"Zero baseline\nP(target)={ba['target_prob']:.3f}, H/H_max={ba['entropy_ratio']:.2f}", fontsize=9)
axes[1].axis("off")

# Blur baseline
blur_img = make_gaussian_blur_baseline(x)
axes[2].imshow(denormalize(blur_img[0]).cpu().permute(1, 2, 0).numpy())
ba = baseline_analysis["Blur baseline"]
axes[2].set_title(f"Blur baseline\nP(target)={ba['target_prob']:.3f}, H/H_max={ba['entropy_ratio']:.2f}", fontsize=9)
axes[2].axis("off")

# N(0,1) sample
noise_sample = torch.randn_like(x)
axes[3].imshow(denormalize(noise_sample[0]).cpu().permute(1, 2, 0).numpy().clip(0, 1))
ba = baseline_analysis["N(0,1) reference"]
axes[3].set_title(f"N(0,1) sample\nE[P(target)]={ba['target_prob']:.3f}, H/H_max={ba['entropy_ratio']:.2f}", fontsize=9)
axes[3].axis("off")

fig.suptitle("Baseline Comparison: What Does Each Method Start From?", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Print analysis
print("\nBaseline OOD Analysis:")
print("─" * 80)
for name, ba in baseline_analysis.items():
    print(f"\n{name}:")
    print(f"  P(target class)  = {ba['target_prob']:.6f}")
    print(f"  Entropy / H_max  = {ba['entropy_ratio']:.4f} (1.0 = uniform = maximally uninformative)")
    print(f"  Top predicted     = {ba['top_class']} ({ba['top_prob']:.4f})")
print("\nIdeal reference: high entropy (close to 1.0), low P(target) -> principled starting point")

---
## 9. Summary Dashboard

All metrics side-by-side with colour-coded winners.

In [ ]:
def build_summary():
    """Collect all metrics into a summary dict."""
    methods = ["Vanilla IG", "Blur IG", "KL-IG"]
    
    summary = {m: {} for m in methods}
    
    for m in methods:
        # Faithfulness
        summary[m]["Ins. AUC"] = id_results[m]["insertion_auc"]
        summary[m]["Del. AUC"] = id_results[m]["deletion_auc"]
        summary[m]["Compl. Err%"] = comp_results[m]["relative_error"] * 100
        
        # Sensitivity
        summary[m]["Occ. Corr"] = occ_corrs[m]["spearman"]
        summary[m]["Gini"] = sparsity_results[m]["gini"]
    
    return summary


summary = build_summary()

# ── Radar chart ──
def radar_chart(summary, methods, metrics, title):
    """Draw a radar chart comparing methods across metrics."""
    n = len(metrics)
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    angles += angles[:1]
    
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    
    for method in methods:
        values = []
        for metric, direction in metrics:
            v = summary[method][metric]
            # Normalise: higher = better for all
            if direction == "lower":
                v = 1.0 / (1.0 + v)  # invert so lower is better
            values.append(v)
        
        # Normalise to [0, 1] across methods
        values_arr = np.array(values)
        values += values[:1]
        ax.plot(angles, values, "o-", linewidth=2, label=method, color=colors[method])
        ax.fill(angles, values, alpha=0.1, color=colors[method])
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([m[0] for m in metrics], fontsize=9)
    ax.set_title(title, fontsize=13, fontweight="bold", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=10)
    
    return fig


metrics_spec = [
    ("Ins. AUC", "higher"),
    ("Del. AUC", "lower"),
    ("Compl. Err%", "lower"),
    ("Occ. Corr", "higher"),
    ("Gini", "higher"),
]

fig = radar_chart(summary, ["Vanilla IG", "Blur IG", "KL-IG"], metrics_spec,
                  "KL-IG Evaluation Summary")
plt.tight_layout()
plt.show()

# ── Final summary table ──
print("\n" + "=" * 85)
print("                    EVALUATION SUMMARY")
print("=" * 85)
print(f"{'Metric':<20} {'Vanilla IG':>14} {'Blur IG':>14} {'KL-IG':>14} {'Winner':>14}")
print("-" * 85)

metric_directions = {
    "Ins. AUC": "max", "Del. AUC": "min", "Compl. Err%": "min",
    "Occ. Corr": "max", "Gini": "max",
}

for metric, direction in metric_directions.items():
    vals = {m: summary[m][metric] for m in ["Vanilla IG", "Blur IG", "KL-IG"]}
    if direction == "max":
        winner = max(vals, key=vals.get)
    else:
        winner = min(vals, key=vals.get)
    
    print(f"{metric:<20} {vals['Vanilla IG']:>14.4f} {vals['Blur IG']:>14.4f} "
          f"{vals['KL-IG']:>14.4f} {'<-- ' + winner:>14}")

print("=" * 85)

---
## 10. Multi-Image Batch Evaluation (Optional)

Uncomment and configure `IMAGE_PATHS` to run the full evaluation across multiple images and aggregate the results.

In [ ]:
# ── Multi-image batch evaluation ──
# Uncomment IMAGE_PATHS below and run this cell to evaluate across many images.

# IMAGE_PATHS = list(Path("/path/to/images").glob("*.jpg"))[:20]

def batch_evaluate(image_paths, model):
    """Run all evaluation metrics across multiple images and aggregate."""
    all_metrics = defaultdict(lambda: defaultdict(list))
    
    for img_path in tqdm(image_paths, desc="Images"):
        try:
            xi = load_image(img_path).to(DEVICE)
            predsi, top_idxi = predict_topk(model, xi)
            targeti = top_idxi[0]
            
            # Compute attributions
            mapsi = {
                "Vanilla IG": compute_vanilla_ig(model, xi, targeti),
                "Blur IG": compute_blur_ig(model, xi, targeti),
            }
            klig_resi = compute_klig(model, xi, targeti)
            mapsi["KL-IG"] = klig_resi.attr_map("absmax")
            
            # Insertion/Deletion
            for name, amap in mapsi.items():
                id_res = insertion_deletion_curves(model, xi, amap, targeti)
                all_metrics[name]["ins_auc"].append(id_res["insertion_auc"])
                all_metrics[name]["del_auc"].append(id_res["deletion_auc"])
            
            # Sparsity
            for name, amap in mapsi.items():
                flat = amap.detach().cpu().numpy().ravel()
                all_metrics[name]["gini"].append(gini_coefficient(flat))
            
            # Occlusion
            _, _, Hi, Wi = xi.shape
            occ_mapi = occlusion_sensitivity(model, xi, targeti)
            for name, amap in mapsi.items():
                attr_grid = attribution_to_patch_grid(amap, OCCLUSION_PATCH_SIZE, OCCLUSION_STRIDE, Hi, Wi)
                corr, _ = stats.spearmanr(occ_mapi.ravel(), attr_grid.ravel())
                all_metrics[name]["occ_corr"].append(corr)
                
        except Exception as e:
            print(f"  Skipping {img_path.name}: {e}")
            continue
    
    # Aggregate
    print("\n" + "=" * 90)
    print("              BATCH EVALUATION SUMMARY (mean +/- std)")
    print("=" * 90)
    print(f"{'Metric':<16} {'Vanilla IG':>20} {'Blur IG':>20} {'KL-IG':>20}")
    print("-" * 90)
    
    for metric in ["ins_auc", "del_auc", "gini", "occ_corr"]:
        row = f"{metric:<16}"
        for method in ["Vanilla IG", "Blur IG", "KL-IG"]:
            vals = all_metrics[method][metric]
            if vals:
                row += f" {np.mean(vals):>8.4f} +/- {np.std(vals):.4f}"
            else:
                row += f" {'N/A':>20}"
        print(row)
    print("=" * 90)
    
    return all_metrics


# Uncomment to run:
# batch_results = batch_evaluate(IMAGE_PATHS, model)

print("Batch evaluation function ready. Uncomment IMAGE_PATHS and run to evaluate.")